In [1]:
import pandas as pd
import re

def process_income(file_path, year):
    df = pd.read_csv(file_path, skiprows=1)

    df = df.rename(columns={
        "Geography": "GEO_ID",
        "Geographic Area Name": "NAME"
    })

    def norm(s):
        s = str(s).lower()
        s = re.sub(r"[^a-z0-9]+", "", s)
        return s

    def find_col(fragment):
        frag = norm(fragment)
        matches = [c for c in df.columns if frag in norm(c)]
        if not matches:
            raise KeyError(f"Missing column containing: {fragment}")
        return matches[0]

    low1 = pd.to_numeric(df[find_col("less than $10,000")], errors="coerce")
    low2 = pd.to_numeric(df[find_col("$10,000 to $14,999")], errors="coerce")
    a = pd.to_numeric(df[find_col("$15,000 to $19,999")], errors="coerce")
    b = pd.to_numeric(df[find_col("$20,000 to $24,999")], errors="coerce")
    c = pd.to_numeric(df[find_col("$25,000 to $29,999")], errors="coerce")
    d = pd.to_numeric(df[find_col("$30,000 to $34,999")], errors="coerce")
    e = pd.to_numeric(df[find_col("$35,000 to $39,999")], errors="coerce")
    f = pd.to_numeric(df[find_col("$40,000 to $44,999")], errors="coerce")
    g = pd.to_numeric(df[find_col("$45,000 to $49,999")], errors="coerce")
    h = pd.to_numeric(df[find_col("$50,000 to $59,999")], errors="coerce")
    i = pd.to_numeric(df[find_col("$60,000 to $74,999")], errors="coerce")
    j = pd.to_numeric(df[find_col("$75,000 to $99,999")], errors="coerce")
    k = pd.to_numeric(df[find_col("$100,000 to $124,999")], errors="coerce")
    l = pd.to_numeric(df[find_col("$125,000 to $149,999")], errors="coerce")
    m = pd.to_numeric(df[find_col("$150,000 to $199,999")], errors="coerce")
    n = pd.to_numeric(df[find_col("$200,000 or more")], errors="coerce")

    df["<15k"] = low1 + low2
    df["15k-25k"] = a + b
    df["25k-35k"] = c + d
    df["35k-50k"] = e + f + g
    df["50k-100k"] = h + i + j
    df["100k-200k"] = k + l + m
    df["200k+"] = n

    df_long = df.melt(
        id_vars=["GEO_ID", "NAME"],
        value_vars=["<15k", "15k-25k", "25k-35k", "35k-50k", "50k-100k", "100k-200k", "200k+"],
        var_name="income",
        value_name="population"
    )

    df_long["year"] = year
    df_long["prop"] = df_long["population"] / df_long.groupby("GEO_ID")["population"].transform("sum")

    return df_long

In [3]:
inc_2021 = process_income(
    r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2021.B19001-Data.csv",
    2021
)

In [5]:
inc_2021.head()

,GEO_ID,NAME,income,population,year,prop
0,0500000US01001,"Autauga County, Alabama",<15k,2264,2021,0.103587
1,0500000US01003,"Baldwin County, Alabama",<15k,7923,2021,0.090871
2,0500000US01005,"Barbour County, Alabama",<15k,1767,2021,0.194432
3,0500000US01007,"Bibb County, Alabama",<15k,1083,2021,0.152901
4,0500000US01009,"Blount County, Alabama",<15k,2669,2021,0.125305


In [7]:
inc_2022 = process_income(
    r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2022.B19001-Data.csv",
    2022
)

In [11]:
inc_2020 = process_income(
    r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2020.B19001-Data.csv",
    2020
)

In [13]:
inc_2023 = process_income(
    r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2023.B19001-Data.csv",
    2023
)

In [15]:
inc_2024 = process_income(
    r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2024.B19001-Data.csv",
    2024
)

In [17]:
inc_all = pd.concat([inc_2020, inc_2021, inc_2022, inc_2023, inc_2024], ignore_index=True)

In [19]:
inc_all.head()

,GEO_ID,NAME,income,population,year,prop
0,0500000US01001,"Autauga County, Alabama",<15k,2313,2020,0.107287
1,0500000US01003,"Baldwin County, Alabama",<15k,8363,2020,0.099504
2,0500000US01005,"Barbour County, Alabama",<15k,2071,2020,0.222163
3,0500000US01007,"Bibb County, Alabama",<15k,1311,2020,0.180603
4,0500000US01009,"Blount County, Alabama",<15k,3113,2020,0.146805


In [21]:
inc_all

,GEO_ID,NAME,income,population,year,prop
0,0500000US01001,"Autauga County, Alabama",<15k,2313,2020,0.107287
1,0500000US01003,"Baldwin County, Alabama",<15k,8363,2020,0.099504
2,0500000US01005,"Barbour County, Alabama",<15k,2071,2020,0.222163
3,0500000US01007,"Bibb County, Alabama",<15k,1311,2020,0.180603
4,0500000US01009,"Blount County, Alabama",<15k,3113,2020,0.146805
...,...,...,...,...,...,...
112744,0500000US72145,"Vega Baja Municipio, Puerto Rico",200k+,304,2024,0.014083
112745,0500000US72147,"Vieques Municipio, Puerto Rico",200k+,22,2024,0.007040
112746,0500000US72149,"Villalba Municipio, Puerto Rico",200k+,105,2024,0.014092
112747,0500000US72151,"Yabucoa Municipio, Puerto Rico",200k+,50,2024,0.004109


In [23]:
inc_all.to_csv("income_all_2020_2024.csv", index=False)